In [2]:
import os
import argparse

from functools import partial

import pyrosettacolabsetup; pyrosettacolabsetup.install_pyrosetta()
import pyrosetta; pyrosetta.init()
from pyrosetta import * 
from pyrosetta.rosetta.core.pose import *
from pyrosetta.rosetta.core.pack.task import *
from pyrosetta.rosetta.protocols import *
from pyrosetta.rosetta.core.select import *

from pyrosetta.rosetta.protocols.simple_moves import MutateResidue
from pyrosetta.rosetta.protocols.relax import FastRelax
from pyrosetta.rosetta.core.kinematics import MoveMap



def load_ligand_params_to_pose(params, pose):
    if len(params) != 0 and params[0] != "":
        params = pyrosetta.Vector1(params)
        res_set = pose.conformation().modifiable_residue_type_set_for_conf()
        res_set.read_files_for_base_residue_types(params)
        pose.conformation().reset_residue_type_set_for_conf(res_set)

def determine_design_repack_residues(
    pose,
    cut1 = 6.0,
    cut2 = 8.0,
    cut3 = 10.0,
    cut4 = 12.0
):
    ligand_atoms = get_ligand_coords(pose)
    designable = list()
    repackable = list()
    
    for i in range(1, pose.total_residue() + 1):
        res = pose.residue(i)
        if not res.is_protein():
            continue

        ca_xyz = res.xyz("CA")
        d_ca = min_distance(ca_xyz, ligand_atoms)

        has_cb = res.has("CB")
        d_cb = None
        if has_cb:
            cb_xyz = res.xyz("CB")
            d_cb = min_distance(cb_xyz, ligand_atoms)

        # Design region
        if d_ca <= cut1:
            designable.append(i)
        elif d_ca <= cut2 and has_cb and d_cb < d_ca:
            designable.append(i)

        # Repack region
        if d_ca <= cut3:
            repackable.append(i)
        elif d_ca <= cut4 and has_cb and d_cb < d_ca:
            repackable.append(i)

    return designable, repackable


def run_fastrelax(pose, scorefxn, repack):

    repack = repack
    
    mm = MoveMap()
    mm.set_bb(False)
    mm.set_chi(False)
    for r in repack:
        mm.set_bb(r, True)
        mm.set_chi(r, True)
        
    for i in range(1, pose.total_residue() + 1):
        if pose.residue(i).is_ligand():
            mm.set_chi(i, True)
    mm.set_jump(False)
    scorefxn = scorefxn.clone()
    scorefxn.set_weight(pyrosetta.rosetta.core.scoring.ScoreType.coordinate_constraint, 1.0)
    
    if not os.getenv("DEBUG"):
        relax = FastRelax(scorefxn, 1)
        relax.constrain_relax_to_start_coords(True)
        relax.max_iter(100)
        # relax.set_scorefxn(scorefxn)
        relax.set_movemap(mm)
        relax.apply(pose)

def get_ligand_coords(pose):
    ligand_atoms = list()

    for i in range(1, pose.total_residue() + 1):
        res = pose.residue(i)
        if res.is_ligand():
            for a in range(1, res.natoms() + 1):
                if res.atom_type(a).is_heavyatom():
                    ligand_atoms.append(res.xyz(a))

    return ligand_atoms

def min_distance(atom_xyz, ligand_coords):
    if atom_xyz is None:
        return None
    else:
        return min((atom_xyz - l).norm() for l in ligand_coords)

┌───────────────────────────────────────────────────────────────────────────────┐
│                                  PyRosetta-4                                  │
│               Created in JHU by Sergey Lyskov and PyRosetta Team              │
│               (C) Copyright Rosetta Commons Member Institutions               │
│                                                                               │
│ NOTE: USE OF PyRosetta FOR COMMERCIAL PURPOSES REQUIRES PURCHASE OF A LICENSE │
│          See LICENSE.PyRosetta.md or email license@uw.edu for details         │
└───────────────────────────────────────────────────────────────────────────────┘
PyRosetta-4 2025 [Rosetta PyRosetta4.Release.python39.ubuntu 2025.45+release.d79cb06334818403e40289334138fb126753c253 2025-11-04T12:19:41] retrieved from: http://www.pyrosetta.org
core.init: Checking for fconfig files in pwd and ./rosetta/flags
core.init: Rosetta version: PyRosetta4.Release.python39.ubuntu r416 2025.45+release.d79cb06334 d79c

In [3]:
input_pdb = "../inputs/rosetta/carA_holo_apa_amp_homologs/carA_A0A010YLP6.pdb"
params = "../inputs/rosetta/subs/LG2.params"

pose = Pose()
load_ligand_params_to_pose(params.split(','), pose)
pyrosetta.io.pose_from_file(pose, input_pdb)

core.conformation.Conformation: [ WARNING ] Attempted to determine the residue type set of an empty pose.
core.chemical.GlobalResidueTypeSet: Finished initializing fa_standard residue type set.  Created 985 residue types
core.chemical.GlobalResidueTypeSet: Total time to initialize 0.67845 seconds.
core.import_pose.import_pose: File '../inputs/rosetta/carA_holo_apa_amp_homologs/carA_A0A010YLP6.pdb' automatically determined to be of type PDB from contents.
core.chemical.GlobalResidueTypeSet: Loading (but possibly not actually using) 'LG2' from the PDB components dictionary for residue type 'pdb_LG2'
core.conformation.Conformation: [ WARNING ] missing heavyatom:  OXT on residue ASP:CtermProteinFull 677


In [4]:
scorefxn = get_fa_scorefxn()
designable, repackable = determine_design_repack_residues(pose)
run_fastrelax(pose, scorefxn, repackable)
pose.dump_pdb(f"carA_A0A010YLP6.relax.pdb")

core.scoring.ScoreFunctionFactory: SCOREFUNCTION: ref2015
core.scoring.etable: Starting energy table calculation
core.scoring.etable: smooth_etable: changing atr/rep split to bottom of energy well
core.scoring.etable: smooth_etable: spline smoothing lj etables (maxdis = 6)
core.scoring.etable: smooth_etable: spline smoothing solvation etables (max_dis = 6)
core.scoring.etable: Finished calculating energy tables.
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBPoly1D.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBFadeIntervals.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/HBEval.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/DonStrength.csv
basic.io.database: Database file opened: scoring/score_functions/hbonds/ref2015_params/AccStrength.csv
basic.io.database: Database file opened: scoring/score_functions/rama/fd/

True